# Practica Hold_Out

![Logo del instituto](img/ITQ.png)


**Hold_Out** Método de validación que divide los datos en dos grupos, uno para entrenar el modelo y otro para evaluar su rendimiento.

**Nombre:** Leticia Verdezoto


**Fecha:** 16-11-2026


**Link:** [Link Repositorio](https://github.com/verdezotoandivel-ctrl/Fundamentos-de-machine-Learning)





In [1]:
import numpy as np
from sklearn import datasets
from sklearn.dummy import DummyClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

### 1. Uso de librerias 

En esta sección importamos las librerias necesarias.

 NumPy: permite realizar operaciones numéricas complejas.

 
 Scikit-learn: proporciona herramientas para trabajar con modelos de aprendizaje automático.

 
 DummyClassifier: permite crear un modelo de referencia.

 
 train_test_split: divide los datos en conjuntos de entrenamiento y prueba.

 
 StandardScaler: estandariza las características.

 
 SVC: implementa un algoritmo de clasificación.

In [2]:
# Carga de datos
iris = datasets.load_iris()
print(iris.target_names)

['setosa' 'versicolor' 'virginica']


### 2. Carga de datos

Cargamos el dataset Iris, que contiene información de 150 flores pertenecientes a tres especies: Setosa, Versicolor y Virginica.

Exploramos sus características para conocer la cantidad de registros, atributos y clases disponibles.

También utilizamos NumPy para contar cuántas instancias pertenecen a cada clase.

In [3]:
# Mostrar características de la tabla de datos.
print("Tabla de datos: %d instancias y %d atributos" % (iris.data.shape[0], iris.data.shape[1]))
print("Valores de la clase:", set(iris.target))

# Cuantificamos el número de instancias que contiene el dataset por clase
valores, ocurrencias = np.unique(iris.target, return_counts=True)
print(valores, ocurrencias)

Tabla de datos: 150 instancias y 4 atributos
Valores de la clase: {np.int64(0), np.int64(1), np.int64(2)}
[0 1 2] [50 50 50]


In [4]:
# Test: hold-out split 80-20%. # Partición externa
X_training, X_test, y_training, y_test = train_test_split(iris.data, iris.target, test_size=0.2, random_state=42)
valores_test, ocur_test = np.unique(y_test, return_counts=True)
print('Test: ', 'clases:', valores_test, ' ocurrencias: ', ocur_test)

Test:  clases: [0 1 2]  ocurrencias:  [10  9 11]


### 3. División de los datos mediante Hold-out

Dividimos el conjunto de datos en dos partes: 80% para entrenamiento y 20% para test.

El conjunto de entrenamiento permite desarrollar el modelo, mientras que el conjunto de test se reserva para evaluar su rendimiento final.

Utilizamos random_state=42 para reproducir la misma división de los datos y stratify para conservar la proporción de las clases.

In [5]:
# Estandarizar las características de entrenamiento y de test
standardizer = StandardScaler()
X_training = standardizer.fit_transform(X_training)
X_test = standardizer.transform(X_test)

### 4. Estandarización de los datos

Aplicamos StandardScaler para transformar las características y conseguir que tengan una escala comparable.

Primero ajustamos el escalador utilizando únicamente los datos de entrenamiento. Después aplicamos esa misma transformación a los conjuntos de validación y test.

Este procedimiento evita la fuga de información y permite que el modelo trabaje con características estandarizadas.

In [6]:
# Validación: hold-out split 80-20%. # Partición interna
X_train, X_val, y_train, y_val = train_test_split(X_training, y_training, test_size=0.2, random_state=42)
valores_train, ocur_train = np.unique(y_train, return_counts=True)
print('Entrenamiento: ', ' clases:', valores_train, '  ocurrencias:', ocur_train)

valores_val, ocur_val = np.unique(y_val, return_counts=True)
print('Validation:    ', ' clases:', valores_val, '  ocurrencias:', ocur_val)

Entrenamiento:   clases: [0 1 2]   ocurrencias: [32 30 34]
Validation:      clases: [0 1 2]   ocurrencias: [ 8 11  5]


In [7]:
# Construcción del objeto que contiene el algoritmo de aprendizaje.
clf = DummyClassifier(strategy='prior', random_state=42)

### 5. Construcción del modelo de referencia

Creamos un modelo DummyClassifier utilizando la estrategia prior.

Este clasificador predice siempre la clase más frecuente del conjunto de entrenamiento.

Su propósito es establecer un rendimiento de referencia que permita comparar los resultados con otros algoritmos de clasificación.

In [8]:
# Entrenamiento del algoritmo de aprendizaje.
clf = clf.fit(X_train, y_train)

In [9]:
# Evaluación del algoritmo de aprendizaje con el método "score" que devuelve directamente la métrica de 'accuracy'
val_accuracy = clf.score(X_val, y_val)
print("Exactitud en validación: ", val_accuracy)

test_accuracy = clf.score(X_test, y_test)
print("Exactitud en test: ", test_accuracy)

Exactitud en validación:  0.20833333333333334
Exactitud en test:  0.36666666666666664


### 6. Evaluación del modelo de referencia

Evaluamos el clasificador utilizando los conjuntos de validación y test mediante el método score().

Esta función calcula la exactitud (Accuracy), que representa la proporción de predicciones correctas respecto al total de registros evaluados.

También obtenemos las predicciones para compararlas con las etiquetas reales y comprobar el comportamiento del clasificador.

In [10]:
# Obtenemos las predicciones sobre conjunto de validación y de test
y_pred_val = clf.predict(X_val)
print('Predicciones de validación ', y_pred_val)
print('Etiquetas reales validación', y_val)

y_pred_test = clf.predict(X_test)
print('\nPredicciones de test ', y_pred_val)
print('Etiquetas reales test', y_val)

Predicciones de validación  [2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2]
Etiquetas reales validación [1 1 0 0 0 2 1 2 2 2 1 1 1 1 1 0 2 0 1 0 1 1 0 0]

Predicciones de test  [2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2]
Etiquetas reales test [1 1 0 0 0 2 1 2 2 2 1 1 1 1 1 0 2 0 1 0 1 1 0 0]


### 7. Comparación entre predicciones y etiquetas reales

Utilizamos el método predict() para obtener las clases que el modelo asigna a cada flor.

Comparamos las predicciones con las etiquetas reales para identificar si el clasificador consigue reconocer correctamente las especies.

Esta comparación permite observar las limitaciones del modelo DummyClassifier.

In [11]:
# Aplicamos un ejemplo con un clasificador más complejo que el "dummyclassifier"
from sklearn.svm import SVC
svc = SVC(C=0.5) # Definimos algoritmo
svc.fit(X_train, y_train) # Entrenamos modelo

val_accuracy = svc.score(X_val, y_val) # Evaluamos modelo en validación
print('Exactitud en validación ', np.round(val_accuracy*100, 4), '%')

test_accuracy = svc.score(X_test, y_test) # Evaluamos modelo en test
print('Exactitud en test ', np.round(test_accuracy*100, 4), '%')

Exactitud en validación  91.6667 %
Exactitud en test  96.6667 %


### 8. Evaluación del modelo SVC

Evaluamos el rendimiento del clasificador SVC utilizando los conjuntos de validación y test.

Calculamos la exactitud mediante score() y expresamos los resultados en porcentajes para facilitar su interpretación.

Finalmente, comparamos el rendimiento de SVC con el modelo de referencia DummyClassifier para determinar cuál consigue clasificar mejor las especies del conjunto Iris.

In [ ]:
# Guardar modelo
import pickle
with open('../models/model.pickle', 'wb') as fw:
    pickle.dump(model, fw)

# Cargar modelo
with open('../models/model.pickle', 'rb') as fr:
    pickle.load(fr)